In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
import os

# ---------- إعداد الجهاز ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

save_folder = "trained_models"
os.makedirs(save_folder, exist_ok=True)
best_model_file = os.path.join(save_folder, "EfficientNetB1_best.pth")
checkpoint_file = os.path.join(save_folder, "EfficientNetB1_checkpoint.pth")



In [ ]:

weights = EfficientNet_B1_Weights.DEFAULT
efficientnet_b1 = efficientnet_b1(weights=weights)

num_classes = 4  # glass, metal, plastic, wood
efficientnet_b1.classifier[1] = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(efficientnet_b1.classifier[1].in_features, num_classes)
)
efficientnet_b1 = efficientnet_b1.to(device)


In [ ]:
if os.path.exists(best_model_file):
    state_dict = torch.load(best_model_file)
    efficientnet_b1.load_state_dict(state_dict, strict=False)  # <-- ignore missing/unexpected keys
    efficientnet_b1.eval()
    print("✅ EfficientNet-B1 loaded (partial) from saved model — ready to use!")
else:
    print("🚀 No saved model found — training can start in next cell.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ---------- Loss + Optimizer ----------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    efficientnet_b1.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ---------- إعداد التدريب ----------
total_epochs = 15
start_epoch = 0
best_val_acc = 0.0

# ---------- لو فيه Checkpoint موجود ----------
if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file)
    efficientnet_b1.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    print(f"🔄 Resuming training from epoch {start_epoch}/{total_epochs} with best_val_acc: {best_val_acc:.4f}")

# ---------- Training Loop ----------
for epoch in range(start_epoch, total_epochs):
    # ---- Training ----
    efficientnet_b1.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = efficientnet_b1(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # ---- Validation ----
    efficientnet_b1.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = efficientnet_b1(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{total_epochs} - Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")

    # ---- حفظ أفضل موديل ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(efficientnet_b1.state_dict(), best_model_file)
        print(f"💾 Best model saved at epoch {epoch+1} with val_acc: {best_val_acc:.4f}")

    # ---- حفظ checkpoint ----
    torch.save({
        'epoch': epoch,
        'model_state': efficientnet_b1.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, checkpoint_file)

efficientnet_b1.eval()
print(f"✅ Training complete! EfficientNet-B1 is ready and saved at {best_model_file}")

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# ---------- إعداد المسارات ----------
save_dir = r"result"
os.makedirs(save_dir, exist_ok=True)
cache_file = os.path.join(save_dir, "test_pred_efficientnetB1.pt")

# ---------- Load cache لو موجود ----------
if os.path.exists(cache_file):
    data = torch.load(cache_file)
    all_preds = data['preds']
    all_labels = data['labels']
    print("✅ Loaded cached test predictions")
else:
    efficientnet_b1.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = efficientnet_b1(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    torch.save({'preds': all_preds, 'labels': all_labels}, cache_file)
    print(f"✅ Test predictions computed and cached at {cache_file}")

# ---------- Confusion Matrix ----------
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ---------- Overall Metrics ----------
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')

print("\n==================== Overall Model Metrics ====================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

# ---------- Per-Class Metrics ----------
report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)

print("\n==================== Per-Class Metrics ====================")
for idx, cls in enumerate(classes):
    # Accuracy لكل كلاس: نسبة العينات المصنفة صح من إجمالي العينات الفعلية للكلاس
    cls_indices = np.where(np.array(all_labels) == idx)[0]
    cls_acc = np.sum(np.array(all_preds)[cls_indices] == np.array(all_labels)[cls_indices]) / len(cls_indices)

    cls_prec = report[cls]['precision']
    cls_rec = report[cls]['recall']

    print(f"\nClass: {cls}")
    print(f"  Accuracy : {cls_acc:.4f}")
    print(f"  Precision: {cls_prec:.4f}")
    print(f"  Recall   : {cls_rec:.4f}")


In [ ]:
# ================== Single Image Prediction for EfficientNet-B1 (auto preprocessing) ==================

import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import os  # <--- عشان نجيب اسم الكلاس من filename

image_path = r"dataset_processed/test/metal/metal_000026.jpg"
image = Image.open(image_path).convert("RGB")

# ---------- Automatic preprocessing ----------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
input_tensor = preprocess(image).unsqueeze(0).to(device)  # Batch 1

# ---------- Prediction ----------
efficientnet_b1.eval()
with torch.no_grad():
    outputs = efficientnet_b1(input_tensor)
    probs = torch.softmax(outputs, dim=1)
    conf, pred = torch.max(probs, 1)

# ---------- Get Actual Class ----------
# نفترض إن اسم الملف بالشكل: <class>_xxxx.jpg
actual_class = os.path.basename(image_path).split('_')[0]

# ---------- Display ----------
plt.imshow(image)
plt.axis('off')
plt.show()

print(f"Predicted Material: {classes[pred.item()]}")
print(f"Confidence: {conf.item():.4f}")
print(f"Actual Material   : {actual_class}")

In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_gradcam(model, input_tensor, target_class=None, device='cuda'):
    model.eval()
    input_tensor = input_tensor.to(device)

    # ---------- تخزين activations و gradients ----------
    activations = None
    gradients = None

    def forward_hook(module, input, output):
        nonlocal activations
        activations = output

    def backward_hook(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0]

    # ---------- اختر آخر Feature Layer في EfficientNet-B1 ----------
    target_layer = model.features[-1]
    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_backward_hook(backward_hook)

    # ---------- Forward ----------
    output = model(input_tensor)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # ---------- Backward ----------
    model.zero_grad()
    loss = output[0, target_class]
    loss.backward()

    # ---------- Grad-CAM ----------
    pooled_grads = torch.mean(gradients, dim=[0, 2, 3])
    for i in range(activations.shape[1]):
        activations[0, i, :, :] *= pooled_grads[i]

    heatmap = torch.sum(activations, dim=1).squeeze().cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / np.max(heatmap)

    # ---------- Superimpose on original image ----------
    img = input_tensor[0].cpu().permute(1,2,0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap)/255
    superimposed_img = heatmap + np.float32(img)
    superimposed_img = superimposed_img / superimposed_img.max()

    plt.figure(figsize=(6,6))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title(f"Grad-CAM for class {classes[target_class]}")
    plt.show()

    # ---------- إزالة الـ hooks ----------
    fh.remove()
    bh.remove()


In [ ]:

show_gradcam(efficientnet_b1, input_tensor, target_class=pred.item(), device=device)